# ETA - XGBoost


## Imports


In [ ]:
import cupy as cp
import numpy as np
import pandas as pd
import xgboost as xgb
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score, train_test_split

## Constants


In [2]:
DATASET_PATH = "../dataset/v2/athens_fcd_output_10hour.csv"
MODEL_PATH = "../models/eta-xgboost.json"

DEVICE = "cuda"
DEVICE

'cuda'

## Load Data


In [3]:
df = pd.read_csv(DATASET_PATH, sep=";")
df

,timestep_time,vehicle_angle,vehicle_id,vehicle_lane,vehicle_pos,vehicle_slope,vehicle_speed,vehicle_type,vehicle_x,vehicle_y
0,0.0,173.49,veh0,449901015_0,5.10,0.0,0.00,DEFAULT_VEHTYPE,472.16,106.95
1,1.0,173.49,veh0,449901015_0,7.12,0.0,2.02,DEFAULT_VEHTYPE,472.39,104.94
2,2.0,173.49,veh0,449901015_0,10.97,0.0,3.85,DEFAULT_VEHTYPE,472.82,101.11
3,3.0,173.49,veh0,449901015_0,17.17,0.0,6.19,DEFAULT_VEHTYPE,473.53,94.96
4,4.0,173.96,veh0,449901015_0,25.88,0.0,8.71,DEFAULT_VEHTYPE,474.44,86.30
...,...,...,...,...,...,...,...,...,...,...
8383189,35999.0,73.63,veh2061,51249283_0,2.10,0.0,0.00,DEFAULT_VEHTYPE,944.83,1450.81
8383190,35999.0,235.79,veh2062,23184187#6_0,2.49,0.0,6.85,DEFAULT_VEHTYPE,1795.44,1071.77
8383191,35999.0,146.88,veh2063,393398563#0_0,57.71,0.0,4.11,DEFAULT_VEHTYPE,1839.79,1180.91
8383192,35999.0,41.53,veh2064,-64094592_0,44.80,0.0,11.74,DEFAULT_VEHTYPE,185.50,1135.05


## Data Cleaning


In [4]:
# Drop unnecessary columns
df = df.drop(columns=["vehicle_lane", "vehicle_slope", "vehicle_type"])

## Feature Engineering


In [5]:
# Sort the DataFrame by vehicle_id and timestep_time
df = df.sort_values(["vehicle_id", "timestep_time"])

# Get destination coordinates and trip duration using groupby-transform
df["destination_x"] = df.groupby("vehicle_id")["vehicle_x"].transform("last")
df["destination_y"] = df.groupby("vehicle_id")["vehicle_y"].transform("last")

# Create a feature for euclidean distance (to destination)
df["euclidean_distance"] = np.hypot(df["destination_x"] - df["vehicle_x"], df["destination_y"] - df["vehicle_y"])

# Create a feature for remaining time (to destination)
df["last_time"] = df.groupby("vehicle_id")["timestep_time"].transform("last")
df["remaining_time"] = df["last_time"] - df["timestep_time"]

# Remove unnecessary columns and rows
df = df.drop(columns=["last_time"])
df = df[df["remaining_time"] > 0].copy()

# Optionally, reset the index
df.reset_index(drop=True, inplace=True)
df

,timestep_time,vehicle_angle,vehicle_id,vehicle_pos,vehicle_speed,vehicle_x,vehicle_y,destination_x,destination_y,euclidean_distance,remaining_time
0,0.0,173.49,veh0,5.10,0.00,472.16,106.95,2190.43,836.39,1866.690790,313.0
1,1.0,173.49,veh0,7.12,2.02,472.39,104.94,2190.43,836.39,1867.265526,312.0
2,2.0,173.49,veh0,10.97,3.85,472.82,101.11,2190.43,836.39,1868.373836,311.0
3,3.0,173.49,veh0,17.17,6.19,473.53,94.96,2190.43,836.39,1870.150811,310.0
4,4.0,173.96,veh0,25.88,8.71,474.44,86.30,2190.43,836.39,1872.767121,309.0
...,...,...,...,...,...,...,...,...,...,...,...
8359419,17150.0,52.20,veh9999,38.87,10.70,1204.45,1154.48,1244.65,1185.66,50.874673,5.0
8359420,17151.0,52.20,veh9999,48.44,9.57,1212.02,1160.35,1244.65,1185.66,41.295436,4.0
8359421,17152.0,52.20,veh9999,59.09,10.65,1220.43,1166.87,1244.65,1185.66,30.654078,3.0
8359422,17153.0,52.20,veh9999,69.64,10.54,1228.77,1173.34,1244.65,1185.66,20.098677,2.0


## Features and Target


In [6]:
# Define the features and target columns
feature_cols = [col for col in df.columns if col not in ["timestep_time", "vehicle_id", "remaining_time"]]
target = "remaining_time"

# Split into X and y
X = df[feature_cols]
y = df[target]

## Train and Test Split


In [7]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check the shape of the data
X_train.shape, X_test.shape

((6687539, 8), (1671885, 8))

## XGBoost Model


In [8]:
# Initialize model
model = xgb.XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device=DEVICE,
    random_state=42,
)

## Training


In [9]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

## Evaluation


In [10]:
# Copy the test set to GPU memory
X_test_gpu = cp.asarray(X_test)

# Predict on the test set
y_pred = model.predict(X_test_gpu)

# Evaluate the model using Mean Absolute Error and Mean Absolute Percentage Error
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"Mean Absolute Error on Test set: {mae:.4f}")
print(f"Mean Absolute Percentage Error on Test set: {mape * 100:.4f}%")

Mean Absolute Error on Test set: 110.8880
Mean Absolute Percentage Error on Test set: 100.9599%


## Hyperparameter Tuning


In [15]:
space = {
    "max_depth": hp.quniform("max_depth", 3, 10, 1),
    "min_child_weight": hp.quniform("min_child_weight", 1, 10, 1),
    "gamma": hp.uniform("gamma", 0, 0.5),
    "subsample": hp.uniform("subsample", 0.5, 1),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.5, 1),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.3),
    "n_estimators": hp.quniform("n_estimators", 100, 500, 50),
}


def objective(params):
    params["max_depth"] = int(params["max_depth"])
    params["min_child_weight"] = int(params["min_child_weight"])
    params["n_estimators"] = int(params["n_estimators"])

    model = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", device=DEVICE, random_state=42, **params)

    cv_score = cross_val_score(model, X_train, y_train, scoring="neg_mean_absolute_error", cv=5, n_jobs=-1)

    loss = -cv_score

    return {
        "loss": loss.mean(),
        "status": STATUS_OK,
    }


trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=10,
    trials=trials,
    rstate=np.random.default_rng(42),
)

best["max_depth"] = int(best["max_depth"])
best["min_child_weight"] = int(best["min_child_weight"])
best["n_estimators"] = int(best["n_estimators"])

print(f"Best hyperparameters: {best}")

100%|██████████| 10/10 [07:11<00:00, 43.19s/trial, best loss: 78.6586570717202]
Best hyperparameters: {'colsample_bytree': np.float64(0.5776778270118749), 'gamma': np.float64(0.4738538693265129), 'learning_rate': np.float64(0.15680113598338938), 'max_depth': 9, 'min_child_weight': 2, 'n_estimators': 500, 'subsample': np.float64(0.8495392973707541)}


In [16]:
best_model = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", device=DEVICE, random_state=42, **best)

best_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.5776778270118749), device='cuda',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None,
             gamma=np.float64(0.4738538693265129), grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=np.float64(0.15680113598338938), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=9, max_leaves=None,
             min_child_weight=2, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [20]:
y_pred = best_model.predict(X_test_gpu)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"Mean Absolute Error on Test set: {mae:.4f}")
print(f"Mean Absolute Percentage Error on Test set: {mape * 100:.4f}%")

Mean Absolute Error on Test set: 78.6377
Mean Absolute Percentage Error on Test set: 76.7041%


## Save Model


In [21]:
# Save the trained model to a file for later use
model.save_model(MODEL_PATH)

## Load Model


In [22]:
# Load the model for making predictions
loaded_model = xgb.XGBRegressor()
loaded_model.load_model(MODEL_PATH)